# Sprint 2 — sparse TF-IDF sweep

Attach the `llm-classification-finetuning` competition data, grant this notebook access to the five `CLEARML_*` Kaggle Secrets, select a CPU session, enable Internet for the Git clone and package installation, then use **Save Version → Save & Run All**. Model selection uses frozen fold 7, not the Kaggle public leaderboard.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPOSITORY = 'https://github.com/kujifined/PMLDL-llm-classification-finetuning.git'
BRANCH = 'experiment/E202609140001-sparse-tfidf-sweep'
REPO_DIR = Path('/kaggle/working/team-repo')
PACKAGES_DIR = Path('/kaggle/working/pmldl-packages')
data_candidates = (
    Path('/kaggle/input/competitions/llm-classification-finetuning'),
    Path('/kaggle/input/llm-classification-finetuning'),
)
DATA_DIR = next((path for path in data_candidates if path.is_dir()), None)
assert DATA_DIR is not None, 'Attach the competition data before running.'
print('Competition data:', DATA_DIR)


In [ ]:
if REPO_DIR.exists():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --branch {BRANCH} --single-branch {REPOSITORY} {REPO_DIR}
%cd {REPO_DIR}
!git status --short --branch


In [ ]:
if PACKAGES_DIR.exists():
    shutil.rmtree(PACKAGES_DIR)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install',
        '--target', str(PACKAGES_DIR),
        '-r', 'requirements-baseline.lock',
        'clearml>=1.17,<3', 'wrapt',
    ],
    cwd=REPO_DIR,
    check=True,
)
verification_environment = os.environ.copy()
verification_environment['PYTHONPATH'] = os.pathsep.join(
    [str(PACKAGES_DIR), str(REPO_DIR / 'src')]
)
verification_environment['PYTHONNOUSERSITE'] = '1'
subprocess.run(
    [
        sys.executable, '-c',
        (
            "import joblib, numpy, pandas, scipy, sklearn, clearml; "
            "print('joblib', joblib.__version__); "
            "print('numpy', numpy.__version__); "
            "print('pandas', pandas.__version__); "
            "print('scipy', scipy.__version__); "
            "print('sklearn', sklearn.__version__); "
            "print('clearml', clearml.__version__)"
        ),
    ],
    env=verification_environment,
    check=True,
)


In [ ]:
from kaggle_secrets import UserSecretsClient

secret_names = (
    'CLEARML_API_ACCESS_KEY',
    'CLEARML_API_SECRET_KEY',
    'CLEARML_API_HOST',
    'CLEARML_WEB_HOST',
    'CLEARML_FILES_HOST',
)
secret_client = UserSecretsClient()
missing = []
for secret_name in secret_names:
    try:
        os.environ[secret_name] = secret_client.get_secret(secret_name)
    except Exception:
        missing.append(secret_name)
assert not missing, f'Missing or ungranted Kaggle Secrets: {missing}'
runtime_environment = os.environ.copy()
runtime_environment['PYTHONPATH'] = os.pathsep.join(
    [str(PACKAGES_DIR), str(REPO_DIR / 'src')]
)
runtime_environment['PYTHONNOUSERSITE'] = '1'
print('ClearML credentials loaded from Kaggle Secrets (values hidden).')


In [ ]:
subprocess.run(
    [
        sys.executable, '-u', 'scripts/sweep_sparse_sprint2.py',
        '--data-dir', str(DATA_DIR), '--external-data',
    ],
    cwd=REPO_DIR,
    env=runtime_environment,
    check=True,
)


In [ ]:
import json
import shutil

run_dirs = sorted(
    (REPO_DIR / 'results' / 'runs').glob('E202609140001__*'),
    key=lambda path: path.stat().st_mtime,
)
assert run_dirs, 'No Sprint 2 run directory was produced.'
run_dir = run_dirs[-1]
run_id = run_dir.name
metrics = json.loads((run_dir / 'metrics.json').read_text())
run_metadata = json.loads((run_dir / 'run.json').read_text())
assert metrics['status'] == 'completed', metrics
artifact_dir = REPO_DIR / 'artifacts' / run_id
bundle_dir = Path('/kaggle/working/nikita_sprint2_handoff')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
shutil.copytree(run_dir, bundle_dir / 'results' / 'runs' / run_id)
shutil.copytree(artifact_dir, bundle_dir / 'artifacts' / run_id)
archive = shutil.make_archive(str(bundle_dir), 'zip', bundle_dir)
print('Run:', run_id)
print('Validation:', metrics['summary']['validation'])
task_id = run_metadata['tracking']['task_id']
if task_id:
    clearml_url = subprocess.check_output(
        [
            sys.executable,
            '-c',
            'import sys; from clearml import Task; print(Task.get_task_output_log_web_page(sys.argv[1]))',
            task_id,
        ],
        text=True,
        env=runtime_environment,
    ).strip().splitlines()[-1]
    print('ClearML:', clearml_url)
else:
    print('ClearML tracking failed:', run_metadata['tracking']['error'])
print('Download:', archive)
